# Verifiable LLM Baseline — GPU Reproducibility (Phase 3) on Google Colab

This notebook extends the **CPU determinism baseline** to a **CUDA GPU**, executing
the Phase 3 goal from the README: *strict GPU determinism* via deterministic cuDNN
and a pinned cuBLAS workspace.

It runs two things on the GPU:
1. **Segmented audit** (`reproducibility.py`) — the 5 falsifiability scenarios. Scenario 1
   (clean replay) must pass bitwise; scenarios 2–5 must be caught.
2. **Fresh-vs-fresh** (`gpu_reproducibility_test.py`) — two from-scratch runs on the
   same GPU must produce bitwise-identical weights.

> **Before you run:** set the runtime to GPU — *Runtime → Change runtime type → Hardware
> accelerator → GPU (T4 is fine)*, then *Runtime → Run all*.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Get the code and pin determinism

Set `REPO_URL` to your fork/repo. (Alternatively, mount Drive or upload the `src/`
folder — see the commented fallbacks.)

`CUBLAS_WORKSPACE_CONFIG` is set here *before* any CUDA op so cuBLAS uses a fixed
reduction order. `src/device.py` also sets it on import, so the scripts are safe even
if you skip this — but setting it now keeps the in-kernel `torch` import deterministic too.

In [ ]:
import os

# Must be set before the first CUDA matmul (read once at CUDA context creation).
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ---- Option A: clone from GitHub (set this to your repo) ----
REPO_URL = "https://github.com/<your-username>/Verifiable-LLM-Baseline.git"
REPO_DIR = "Verifiable-LLM-Baseline"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 $REPO_URL $REPO_DIR

# ---- Option B (fallback): mount Google Drive and point REPO_DIR at your copy ----
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_DIR = '/content/drive/MyDrive/Verifiable-LLM-Baseline'

# ---- Option C (fallback): upload a zip of the repo ----
# from google.colab import files; files.upload()   # then: !unzip -q Verifiable-LLM-Baseline.zip

SRC_DIR = os.path.join(REPO_DIR, "src")
assert os.path.isdir(SRC_DIR), f"src/ not found at {SRC_DIR} — fix REPO_URL or use a fallback."

# Colab ships a CUDA-enabled torch already; only ensure the light deps.
!pip install -q "numpy==2.4.3" "tqdm==4.67.3"
print("src ready at:", SRC_DIR)

## 3. Environment fingerprint

Determinism guarantees hold **within one GPU model**. The device name below is part of
the trust anchor — a different GPU (or a different cuBLAS/cuDNN version) may produce a
different, but still internally reproducible, set of bits.

In [ ]:
import torch
print("torch           :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("CUDA runtime    :", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("cuDNN          :", torch.backends.cudnn.version())
print("CUBLAS workspace:", os.environ.get("CUBLAS_WORKSPACE_CONFIG"))
assert torch.cuda.is_available(), "No GPU — set Runtime → Change runtime type → GPU."

## 4. Phase 3 — segmented audit on GPU (5 scenarios)

Expected: **Scenario 1 PASSES** (clean replay is bitwise deterministic on this GPU);
**Scenarios 2–5 FAIL** (wrong seed, injected gradient noise, post-training sabotage,
and a tampered checkpoint file are all detected).

In [ ]:
!cd "$SRC_DIR" && python reproducibility.py

## 5. Fresh-vs-fresh — bitwise GPU reproducibility

Trains from scratch twice on this GPU and checks the two runs are bitwise identical.
Appends a proof block to `proofs/device_determinism_log.txt`.

In [ ]:
!cd "$SRC_DIR" && python gpu_reproducibility_test.py

## 6. Eval + sealed global manifest

Runs the deterministic held-out eval and seals the end-to-end pipeline hash
(environment + config + dataset + checkpoint + eval).

In [ ]:
!cd "$SRC_DIR" && python eval.py && python global_manifest.py

## Interpreting the results

- **Same GPU → same bits.** A passing Scenario 1 and a passing fresh-vs-fresh test show
  the software entropy is fully controlled on this device.
- **Different GPU → different (but reproducible) bits.** Checkpoint hashes produced on a
  T4 will not match those from an A100 or from CPU. That cross-hardware drift is the
  Phase 2 quantity this baseline is built to measure — not a failure.
- **If you hit a `nondeterministic ... CUDA` error**, an op without a deterministic
  kernel was reached under `torch.use_deterministic_algorithms(True)`. That is a genuine
  finding worth recording — it pinpoints exactly where hardware entropy enters.
- **RNG coverage:** GPU dropout draws from the CUDA generator, so the checkpoint now
  serializes CUDA RNG state alongside CPU/NumPy/Python state — without it, a resumed
  replay would diverge at the first dropout mask.